In [1]:
import os, glob, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from tqdm import tqdm

In [ ]:
KAGGLE_BASE  = '/kaggle/input/datasets/qwerte123/hetero-data-updated-diffemb'
LOCAL_BASE   = r'C:\Users\Admin\Documents\GitHub\claude_plum\data'
BASE         = KAGGLE_BASE if os.path.exists(KAGGLE_BASE) else LOCAL_BASE

DATA_PATH     = os.path.join(BASE, 'heterodata_object12_updated.pt')
CKPT_DIR_IMP  = os.path.join(BASE, 'checkpoints_improved')
CKPT_DIR_ANTI = os.path.join(BASE, 'checkpoints_anti')

SAVE_DIR      = ('/kaggle/working/checkpoints_gpt2' if os.path.exists('/kaggle')
                 else r'C:\Users\Admin\Documents\GitHub\claude_plum\data\checkpoints_gpt2')
SAVE_DIR_ANTI = ('/kaggle/working/checkpoints_gpt2_anti' if os.path.exists('/kaggle')
                 else r'C:\Users\Admin\Documents\GitHub\claude_plum\data\checkpoints_gpt2_anti')
os.makedirs(SAVE_DIR,      exist_ok=True)
os.makedirs(SAVE_DIR_ANTI, exist_ok=True)

device  = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')

df      = torch.load(DATA_PATH, weights_only=False, map_location='cpu')
embeds  = df['item'].x
n_items = embeds.shape[0]
print(f'Items: {n_items}  Embed dim: {embeds.shape[1]}')
print(df)

## 1. Load RQ-VAE and assign SIDs to all items

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()
        layers, dims = [], [input_dim] + list(hidden_dims) + [output_dim]
        for i, (a, b) in enumerate(zip(dims[:-1], dims[1:])):
            layers.append(nn.Linear(a, b, bias=True))
            if i < len(dims) - 2:
                layers.append(nn.LayerNorm(b))
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)


class EMACodebook(nn.Module):
    def __init__(self, codebook_size, emb_dim, beta=0.25, ema_decay=0.99, epsilon=1e-4):
        super().__init__()
        self.codebook_size = codebook_size
        self.beta = beta; self.ema_decay = ema_decay; self.epsilon = epsilon
        emb = F.normalize(torch.randn(codebook_size, emb_dim), p=2, dim=1)
        self.register_buffer('emb',         emb)
        self.register_buffer('ema_count',   torch.ones(codebook_size))
        self.register_buffer('ema_weight',  emb.clone())
        self.register_buffer('initialized', torch.zeros(1, dtype=torch.bool))
    def forward(self, x):
        x_n    = F.normalize(x,        p=2, dim=1)
        code_n = F.normalize(self.emb, p=2, dim=1)
        ids    = (1.0 - x_n @ code_n.T).argmin(dim=1)
        emb    = self.emb[ids]
        return self.beta * F.mse_loss(x, emb.detach()), x + (emb - x).detach(), ids


class _RQVAEBase(nn.Module):
    """Shared RQ-VAE inference logic. Subclasses differ only in training loss."""
    def __init__(self, inp_size, hidden_sizes, embed_dim, n_layers,
                 codebook_size=256, beta=0.25, gamma=0.1, ema_decay=0.99, **kwargs):
        super().__init__()
        self.n_layers = n_layers
        self.enc = Encoder(inp_size, hidden_sizes, embed_dim)
        self.dec = Encoder(embed_dim, hidden_sizes[::-1], inp_size)
        self.codebooks = nn.ModuleList([
            EMACodebook(codebook_size, embed_dim, beta=beta, ema_decay=ema_decay)
            for _ in range(n_layers)
        ])

    def forward(self, x):
        x_n = F.normalize(x, p=2, dim=1)
        r, sids = self.enc(x_n), []
        for cb in self.codebooks:
            _, emb_st, ids = cb(r)
            r = r - emb_st.detach()
            sids.append(ids)
        return {'sids': sids}


class RQVAE_Improved(_RQVAEBase):
    def __init__(self, *args, temperature=0.07, **kwargs):
        super().__init__(*args, **kwargs)
        self.temperature = temperature

class RQVAE_AntiContrastive(_RQVAEBase):
    def __init__(self, *args, margin=0.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.margin = margin

In [ ]:
import re as _re

def infer_hidden_sizes(state_dict):
    """Infer encoder hidden_sizes from checkpoint without needing them in hparams.
    Must sort numerically (not lexicographically) to handle indices like 9, 10, 12.
    """
    items = [(k, v) for k, v in state_dict.items()
             if k.startswith('enc.net.') and k.endswith('.weight') and v.dim() == 2]
    items.sort(key=lambda kv: int(_re.search(r'enc\.net\.(\d+)\.weight', kv[0]).group(1)))
    outs = [v.shape[0] for _, v in items]
    return outs[:-1]  # exclude final embed_dim output layer

def load_rqvae(ckpt_path, model_class, inp_size, device):
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    hp   = ckpt['hparams']
    hs   = infer_hidden_sizes(ckpt['model_state'])
    kw   = {k: hp[k] for k in ('embed_dim','n_layers','codebook_size','beta','gamma','ema_decay') if k in hp}
    if 'temperature' in hp: kw['temperature'] = hp['temperature']
    if 'margin'      in hp: kw['margin']      = hp['margin']
    model = model_class(inp_size=inp_size, hidden_sizes=hs, **kw).to(device)
    model.load_state_dict(ckpt['model_state'])
    model.eval()
    print(f'  {model_class.__name__}  layers={hp["n_layers"]}  '
          f'codebook={hp["codebook_size"]}  embed={hp["embed_dim"]}  hidden={hs}')
    return model, hp

# ── Load improved ──────────────────────────────────────────────────────────────
ckpts_imp = sorted(glob.glob(os.path.join(CKPT_DIR_IMP, 'rqvae_improved_s*.pt')))
assert ckpts_imp, f'No checkpoint in {CKPT_DIR_IMP}'
print(f'Loading: {ckpts_imp[-1]}')
rqvae_imp, hp_imp = load_rqvae(ckpts_imp[-1], RQVAE_Improved, embeds.shape[1], device)

# ── Load anti-contrastive (optional) ──────────────────────────────────────────
ckpts_anti = sorted(glob.glob(os.path.join(CKPT_DIR_ANTI, 'rqvae_anti_s*.pt')))
rqvae_anti, hp_anti = None, None
if ckpts_anti:
    print(f'Loading: {ckpts_anti[-1]}')
    rqvae_anti, hp_anti = load_rqvae(ckpts_anti[-1], RQVAE_AntiContrastive, embeds.shape[1], device)
else:
    print('WARNING: No anti checkpoint found — anti model will be skipped.')

In [ ]:
PAD_ID = 0
BOS_ID = 1

def assign_sids(rqvae_model, embeds, device):
    """Assign SIDs from model + add a disambiguation suffix for any collisions.
    Returns: item_sids (dict), sid_to_item (dict), max_dupe (int).
    """
    n = embeds.shape[0]
    base = {}
    rqvae_model.eval()
    with torch.no_grad():
        for i in range(n):
            out     = rqvae_model(embeds[i].unsqueeze(0).to(device))
            base[i] = tuple(s.item() for s in out['sids'])

    ctr, full = defaultdict(int), {}
    for i in range(n):
        s = base[i]
        full[i] = s + (ctr[s],)
        ctr[s] += 1

    max_dupe      = max(ctr.values())
    n_base_unique = len(set(base.values()))
    sid_to_item   = defaultdict(list)
    for i, sid in full.items():
        sid_to_item[sid].append(i)
    assert len(sid_to_item) == n

    print(f'  base unique: {n_base_unique}/{n}  collisions: {n - n_base_unique}  '
          f'max_dupe: {max_dupe}  → {n} unique tuples')
    return full, sid_to_item, max_dupe

def make_vocab(hp, max_dupe):
    """Compute tokenization constants from model hparams."""
    L, K       = hp['n_layers'], hp['codebook_size']
    n_levels   = L + 1
    lev_off    = [2 + l * K for l in range(L)] + [2 + L * K]
    vocab_size = 2 + L * K + max_dupe
    return n_levels, lev_off, vocab_size

def make_tokenizer(item_sids, lev_off, n_levels):
    def item_to_tokens(item_id):
        sid = item_sids[item_id]
        return [sid[l] + lev_off[l] for l in range(n_levels)]
    def history_to_tokens(item_ids):
        toks = [BOS_ID]
        for iid in item_ids: toks.extend(item_to_tokens(int(iid)))
        return toks
    return item_to_tokens, history_to_tokens

def build_trie(sid_to_item):
    """Nested dict trie. Leaves: {last_code: item_id}."""
    trie = {}
    for sid, ids in sid_to_item.items():
        node = trie
        for c in sid[:-1]: node = node.setdefault(c, {})
        node[sid[-1]] = ids[0]
    return trie

# ── Improved ──────────────────────────────────────────────────────────────────
print('RQVAE_Improved:')
sids_imp,  sid2item_imp,  max_dupe_imp  = assign_sids(rqvae_imp, embeds, device)
n_lev_imp, lev_off_imp,   vocab_imp     = make_vocab(hp_imp,  max_dupe_imp)
tok_imp,   hist_tok_imp                  = make_tokenizer(sids_imp, lev_off_imp, n_lev_imp)
trie_imp                                 = build_trie(sid2item_imp)
print(f'  vocab={vocab_imp}  n_levels={n_lev_imp}  lev_off={lev_off_imp}')

# ── Anti ──────────────────────────────────────────────────────────────────────
if rqvae_anti is not None:
    print('RQVAE_Anti:')
    sids_anti, sid2item_anti, max_dupe_anti = assign_sids(rqvae_anti, embeds, device)
    n_lev_anti, lev_off_anti, vocab_anti    = make_vocab(hp_anti, max_dupe_anti)
    tok_anti,   hist_tok_anti               = make_tokenizer(sids_anti, lev_off_anti, n_lev_anti)
    trie_anti                               = build_trie(sid2item_anti)
    print(f'  vocab={vocab_anti}  n_levels={n_lev_anti}  lev_off={lev_off_anti}')

In [14]:
hist = df['user', 'rated', 'item'].history

def make_train_split():
    """Full variable-length histories from training set."""
    item_ids  = hist['train']['item_ID']        
    item_next = hist['train']['item_ID_next']  
    samples = []
    for u in range(len(item_next)):
        ctx = [int(x) for x in item_ids[u] if int(x) >= 0]
        tgt = int(item_next[u])
        if not ctx or tgt < 0 or tgt >= n_items: continue
        samples.append((ctx, tgt))
    return samples

def make_eval_split(split_key):
    """Padded histories from valid / test sets."""
    item_ids  = hist[split_key]['item_ID']       
    item_next = hist[split_key]['item_ID_next']   
    samples = []
    for u in range(len(item_next)):
        ctx = [int(x) for x in item_ids[u].tolist() if int(x) >= 0]
        tgt = int(item_next[u])
        if not ctx or tgt < 0 or tgt >= n_items: continue
        samples.append((ctx, tgt))
    return samples

samples_train = make_train_split()
samples_val   = make_eval_split('valid')
samples_test  = make_eval_split('test')

print(f'Train samples : {len(samples_train)}')
print(f'Val   samples : {len(samples_val)}')
print(f'Test  samples : {len(samples_test)}')

ctx_lens = [len(s[0]) for s in samples_train]
print(f'\nTrain context length — min:{min(ctx_lens)}  max:{max(ctx_lens)}  '
      f'mean:{sum(ctx_lens)/len(ctx_lens):.1f}')

Train samples : 22363
Val   samples : 22363
Test  samples : 22363

Train context length — min:3  max:202  mean:6.9


In [15]:
class GPT2Rec(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, max_seq_len,
                 n_levels, dropout=0.1):
        super().__init__()
        self.d_model     = d_model
        self.n_levels    = n_levels
        self.max_seq_len = max_seq_len

        self.tok_emb = nn.Embedding(vocab_size,   d_model, padding_idx=PAD_ID)
        self.pos_emb = nn.Embedding(max_seq_len,  d_model)
        # Level embedding: which codebook level this token belongs to (0..n_levels-1)
        # BOS gets level id 0 by convention.
        self.lvl_emb = nn.Embedding(n_levels, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=4 * d_model,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.ln_f    = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight  # weight tying

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)
                if m.padding_idx is not None:
                    m.weight.data[m.padding_idx].zero_()

    def _level_ids(self, seq_len, device):
        """Return level ids [0, 0,1,2,3, 0,1,2,3, ...] for a sequence of length seq_len."""
        ids = torch.zeros(seq_len, dtype=torch.long, device=device)
        for pos in range(1, seq_len):
            ids[pos] = (pos - 1) % self.n_levels
        return ids.unsqueeze(0)  
    def forward(self, input_ids):
        B, T = input_ids.shape
        pos  = torch.arange(T, device=input_ids.device).unsqueeze(0)  
        lvl  = self._level_ids(T, input_ids.device).expand(B, -1)     

        x    = self.tok_emb(input_ids) + self.pos_emb(pos) + self.lvl_emb(lvl)

        causal = nn.Transformer.generate_square_subsequent_mask(T, device=input_ids.device)
        pad_m  = (input_ids == PAD_ID)  
        for layer in self.transformer.layers:
            x = layer(x, src_mask=causal, src_key_padding_mask=pad_m)
            x = x.masked_fill(pad_m.unsqueeze(-1), 0.0)

        if self.transformer.norm is not None:
            x = self.transformer.norm(x)
        x = self.ln_f(x)
        return self.lm_head(x)  

In [ ]:
class RecDataset(Dataset):
    """One dataset class for all models — tokenizer functions passed as arguments."""
    def __init__(self, samples, max_hist_len, n_levels, full_supervision,
                 item_to_tokens, history_to_tokens):
        self.samples           = samples
        self.max_hist_len      = max_hist_len
        self.n_levels          = n_levels
        self.full_supervision  = full_supervision
        self.item_to_tokens    = item_to_tokens
        self.history_to_tokens = history_to_tokens

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        ctx, tgt = self.samples[idx]
        ctx = ctx[-self.max_hist_len:]
        inp = self.history_to_tokens(ctx) + self.item_to_tokens(tgt)
        inp = torch.tensor(inp, dtype=torch.long)
        lbl = inp[1:].clone()
        if not self.full_supervision:
            lbl[:-self.n_levels] = -100
        lbl = torch.cat([lbl, torch.tensor([-100])])
        return inp, lbl


def collate_fn(batch):
    inps, lbls = zip(*batch)
    max_len    = max(x.shape[0] for x in inps)
    pad_inp, pad_lbl = [], []
    for inp, lbl in zip(inps, lbls):
        pad = max_len - inp.shape[0]
        pad_inp.append(F.pad(inp, (pad, 0), value=PAD_ID))
        pad_lbl.append(F.pad(lbl, (pad, 0), value=-100))
    return torch.stack(pad_inp), torch.stack(pad_lbl)

In [ ]:
MAX_HIST_LEN = 20
D_MODEL      = 256
N_HEADS      = 8
N_LAYERS     = 4
DROPOUT      = 0.1
BATCH_SIZE   = 256
LR           = 1e-3
WARMUP_STEPS = 500
N_EPOCHS     = 50
BEAM_SIZE    = 32
EVAL_KS      = [1, 5, 10, 20]

def make_loaders(samples_tr, samples_va, n_levels, item_to_tokens, history_to_tokens):
    max_seq = 1 + (MAX_HIST_LEN + 1) * n_levels
    ds_tr = RecDataset(samples_tr, MAX_HIST_LEN, n_levels, True,  item_to_tokens, history_to_tokens)
    ds_va = RecDataset(samples_va, MAX_HIST_LEN, n_levels, False, item_to_tokens, history_to_tokens)
    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, num_workers=0)
    dl_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)
    print(f'  Train: {len(ds_tr)}  Val: {len(ds_va)}  max_seq_len: {max_seq}')
    return dl_tr, dl_va, max_seq

def make_gpt2rec(vocab_size, n_levels, max_seq_len):
    m = GPT2Rec(vocab_size=vocab_size, d_model=D_MODEL, n_heads=N_HEADS,
                n_layers=N_LAYERS, max_seq_len=max_seq_len,
                n_levels=n_levels, dropout=DROPOUT).to(device)
    print(f'  GPT2Rec params: {sum(p.numel() for p in m.parameters() if p.requires_grad):,}')
    return m

def make_optimizer(model, n_batches):
    opt   = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total = N_EPOCHS * n_batches
    sch   = optim.lr_scheduler.LambdaLR(opt, lambda s: (
        s / max(1, WARMUP_STEPS) if s < WARMUP_STEPS
        else max(0.05, 0.5 * (1.0 + math.cos(
            math.pi * (s - WARMUP_STEPS) / max(1, total - WARMUP_STEPS))))
    ))
    return opt, sch

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device, vocab_size):
    model.train()
    total, n = 0.0, 0
    for inp, lbl in tqdm(loader, leave=False, desc='train'):
        inp, lbl = inp.to(device), lbl.to(device)
        loss = F.cross_entropy(model(inp).view(-1, vocab_size), lbl.view(-1), ignore_index=-100)
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total += loss.item(); n += 1
    return total / n

def val_loss(model, loader, device, vocab_size):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for inp, lbl in loader:
            inp, lbl = inp.to(device), lbl.to(device)
            total += F.cross_entropy(model(inp).view(-1, vocab_size),
                                     lbl.view(-1), ignore_index=-100).item()
            n += 1
    return total / n

In [ ]:
# ── Improved: loaders + model + optimizer ─────────────────────────────────────
print('RQVAE_Improved GPT2Rec:')
dl_tr_imp, dl_va_imp, max_seq_imp = make_loaders(
    samples_train, samples_val, n_lev_imp, tok_imp, hist_tok_imp)
gpt2_imp = make_gpt2rec(vocab_imp, n_lev_imp, max_seq_imp)
opt_imp, sch_imp = make_optimizer(gpt2_imp, len(dl_tr_imp))

# ── Training loop ─────────────────────────────────────────────────────────────
best_val_imp = float('inf')
history_imp  = defaultdict(list)

for epoch in range(1, N_EPOCHS + 1):
    tr = train_epoch(gpt2_imp, dl_tr_imp, opt_imp, sch_imp, device, vocab_imp)
    va = val_loss(gpt2_imp,   dl_va_imp, device, vocab_imp)
    history_imp['train_loss'].append(tr)
    history_imp['val_loss'].append(va)

    if va < best_val_imp:
        best_val_imp = va
        torch.save({
            'model_state': gpt2_imp.state_dict(),
            'epoch': epoch, 'val_loss': best_val_imp,
            'hparams': dict(
                vocab_size=vocab_imp, d_model=D_MODEL, n_heads=N_HEADS,
                n_layers=N_LAYERS, max_seq_len=max_seq_imp, n_levels=n_lev_imp,
                dropout=DROPOUT, rqvae_codebook_size=hp_imp['codebook_size'],
                rqvae_n_layers=hp_imp['n_layers'],
            ),
        }, os.path.join(SAVE_DIR, 'gpt2rec_imp_best.pt'))

    if epoch % 5 == 0:
        print(f'[Imp] Epoch {epoch:3d}/{N_EPOCHS}  '
              f'train={tr:.4f}  val={va:.4f}  '
              f'lr={sch_imp.get_last_lr()[0]:.2e}  best={best_val_imp:.4f}')

print('Improved training complete.')

In [ ]:
@torch.no_grad()
def beam_search(model, ctx_tok, trie, beam_size, device, level_offsets):
    """Generic N-level constrained beam search over SID trie.
    Works for any number of levels — inferred from len(level_offsets).
    Trie leaves: {last_code: item_id (int)}.
    """
    model.eval()
    n_levels = len(level_offsets)
    ctx      = ctx_tok.to(device)

    # Level 0: score all root codes
    logits0 = model(ctx.unsqueeze(0))[0, -1, :]
    beams   = [(logits0[c + level_offsets[0]].item(), (c,), sub)
               for c, sub in trie.items()]
    beams.sort(key=lambda x: -x[0])
    beams = beams[:beam_size]

    for lvl in range(1, n_levels):
        # Batch: append current codes to context for all beams
        code_toks = torch.tensor(
            [[codes[l] + level_offsets[l] for l in range(len(codes))]
             for _, codes, _ in beams],
            device=device,
        )
        batch  = torch.cat([ctx.unsqueeze(0).expand(len(beams), -1), code_toks], dim=1)
        logits = model(batch)[:, -1, :]

        new_beams = []
        is_last   = (lvl == n_levels - 1)
        for i, (score, codes, node) in enumerate(beams):
            for c, child in node.items():
                ns = score + logits[i, c + level_offsets[lvl]].item()
                # Last level: child is item_id (int), not a sub-trie
                new_beams.append((ns, child) if is_last else (ns, codes + (c,), child))
        new_beams.sort(key=lambda x: -x[0])
        if is_last:
            return new_beams  # list of (score, item_id)
        beams = new_beams[:beam_size]

    return []

In [ ]:
def evaluate(samples, model, trie, level_offsets, history_to_tokens,
             beam_size, ks, device, max_hist_len, desc='eval'):
    hits, ndcg, total = defaultdict(int), defaultdict(float), 0
    for ctx, tgt in tqdm(samples, desc=desc, leave=False):
        ctx_tok    = torch.tensor(history_to_tokens(ctx[-max_hist_len:]), dtype=torch.long)
        ranked     = beam_search(model, ctx_tok, trie, beam_size, device, level_offsets)
        ranked_ids = [iid for _, iid in ranked]
        for k in ks:
            top_k = ranked_ids[:k]
            if tgt in top_k:
                hits[k] += 1
                ndcg[k] += 1.0 / math.log2(top_k.index(tgt) + 2)
        total += 1
    return {**{f'Recall@{k}': round(hits[k]/total, 4) for k in ks},
            **{f'NDCG@{k}':   round(ndcg[k]/total, 4) for k in ks},
            'n_users': total}

# ── Load best improved checkpoint and evaluate ─────────────────────────────────
best_ckpt_imp = torch.load(os.path.join(SAVE_DIR, 'gpt2rec_imp_best.pt'),
                            map_location=device, weights_only=False)
gpt2_imp.load_state_dict(best_ckpt_imp['model_state'])
print(f'Loaded improved checkpoint  epoch={best_ckpt_imp["epoch"]}  '
      f'val={best_ckpt_imp["val_loss"]:.4f}')

print('Evaluating improved...')
val_imp  = evaluate(samples_val,  gpt2_imp, trie_imp, lev_off_imp, hist_tok_imp,
                    BEAM_SIZE, EVAL_KS, device, MAX_HIST_LEN, 'val-imp')
test_imp = evaluate(samples_test, gpt2_imp, trie_imp, lev_off_imp, hist_tok_imp,
                    BEAM_SIZE, EVAL_KS, device, MAX_HIST_LEN, 'test-imp')
print(pd.DataFrame({'Val': val_imp, 'Test': test_imp}).T.to_string())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('GPT2Rec — Training history & results', fontsize=13)

# ── Loss curves ───────────────────────────────────────────────────────────────
ax = axes[0]
ep = range(1, len(history_imp['train_loss']) + 1)
ax.plot(ep, history_imp['train_loss'], label='Imp train')
ax.plot(ep, history_imp['val_loss'],   label='Imp val', ls='--')
if rqvae_anti is not None and history_anti['train_loss']:
    ep_a = range(1, len(history_anti['train_loss']) + 1)
    ax.plot(ep_a, history_anti['train_loss'], label='Anti train')
    ax.plot(ep_a, history_anti['val_loss'],   label='Anti val', ls='--')
ax.set_title('Loss'); ax.legend(fontsize=7); ax.set_yscale('log'); ax.set_xlabel('Epoch')

# ── Recall@K ─────────────────────────────────────────────────────────────────
rec_keys = [f'Recall@{k}' for k in EVAL_KS]
x = np.arange(len(rec_keys)); w = 0.35
axes[1].bar(x - w/2, [test_imp[k] for k in rec_keys], w, label='Improved',  color='steelblue')
if rqvae_anti is not None:
    axes[1].bar(x + w/2, [test_anti[k] for k in rec_keys], w, label='Anti', color='orange')
axes[1].set_xticks(x); axes[1].set_xticklabels(rec_keys)
axes[1].set_title('Test Recall@K'); axes[1].legend()

# ── NDCG@K ───────────────────────────────────────────────────────────────────
ndcg_keys = [f'NDCG@{k}' for k in EVAL_KS]
axes[2].bar(x - w/2, [test_imp[k] for k in ndcg_keys], w, label='Improved',  color='steelblue')
if rqvae_anti is not None:
    axes[2].bar(x + w/2, [test_anti[k] for k in ndcg_keys], w, label='Anti', color='orange')
axes[2].set_xticks(x); axes[2].set_xticklabels(ndcg_keys)
axes[2].set_title('Test NDCG@K'); axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'gpt2rec_results.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
if rqvae_anti is None:
    print('No anti model — skipping anti GPT2Rec training.')
else:
    # ── Anti: loaders + model + optimizer ─────────────────────────────────────
    print('RQVAE_Anti GPT2Rec:')
    dl_tr_anti, dl_va_anti, max_seq_anti = make_loaders(
        samples_train, samples_val, n_lev_anti, tok_anti, hist_tok_anti)
    gpt2_anti = make_gpt2rec(vocab_anti, n_lev_anti, max_seq_anti)
    opt_anti, sch_anti = make_optimizer(gpt2_anti, len(dl_tr_anti))

    # ── Training loop ──────────────────────────────────────────────────────────
    best_val_anti = float('inf')
    history_anti  = defaultdict(list)

    for epoch in range(1, N_EPOCHS + 1):
        tr = train_epoch(gpt2_anti, dl_tr_anti, opt_anti, sch_anti, device, vocab_anti)
        va = val_loss(gpt2_anti,   dl_va_anti, device, vocab_anti)
        history_anti['train_loss'].append(tr)
        history_anti['val_loss'].append(va)

        if va < best_val_anti:
            best_val_anti = va
            torch.save({
                'model_state': gpt2_anti.state_dict(),
                'epoch': epoch, 'val_loss': best_val_anti,
                'hparams': dict(
                    vocab_size=vocab_anti, d_model=D_MODEL, n_heads=N_HEADS,
                    n_layers=N_LAYERS, max_seq_len=max_seq_anti, n_levels=n_lev_anti,
                    dropout=DROPOUT, rqvae_codebook_size=hp_anti['codebook_size'],
                    rqvae_n_layers=hp_anti['n_layers'],
                ),
            }, os.path.join(SAVE_DIR_ANTI, 'gpt2rec_anti_best.pt'))

        if epoch % 5 == 0:
            print(f'[Anti] Epoch {epoch:3d}/{N_EPOCHS}  '
                  f'train={tr:.4f}  val={va:.4f}  '
                  f'lr={sch_anti.get_last_lr()[0]:.2e}  best={best_val_anti:.4f}')

    print('Anti training complete.')

In [ ]:
if rqvae_anti is None:
    print('No anti model — skipping evaluation.')
else:
    best_ckpt_anti = torch.load(os.path.join(SAVE_DIR_ANTI, 'gpt2rec_anti_best.pt'),
                                 map_location=device, weights_only=False)
    gpt2_anti.load_state_dict(best_ckpt_anti['model_state'])
    print(f'Loaded anti checkpoint  epoch={best_ckpt_anti["epoch"]}  '
          f'val={best_ckpt_anti["val_loss"]:.4f}')

    print('Evaluating anti...')
    val_anti  = evaluate(samples_val,  gpt2_anti, trie_anti, lev_off_anti, hist_tok_anti,
                         BEAM_SIZE, EVAL_KS, device, MAX_HIST_LEN, 'val-anti')
    test_anti = evaluate(samples_test, gpt2_anti, trie_anti, lev_off_anti, hist_tok_anti,
                         BEAM_SIZE, EVAL_KS, device, MAX_HIST_LEN, 'test-anti')

    print('\n=== Improved vs Anti-Contrastive ===')
    print(pd.DataFrame({
        'Imp Val':   val_imp,
        'Imp Test':  test_imp,
        'Anti Val':  val_anti,
        'Anti Test': test_anti,
    }).T.to_string())

    metrics_keys = [f'Recall@{k}' for k in EVAL_KS] + [f'NDCG@{k}' for k in EVAL_KS]
    delta = {k: round(test_anti[k] - test_imp[k], 4) for k in metrics_keys}
    print('\n=== Test delta (Anti − Improved) ===')
    print(pd.Series(delta).to_string())